# 07 - Track A: CESNET-TLS22 service-category classifier

**Purpose:** Train a gradient-boosted classifier on flow metadata to predict the dataset's own service category, split by day.

**Inputs:** CESNET-TLS22 S HDF5 (bounded deterministic sample read in chunks with h5py)

**Outputs:** metrics vs majority and rule-based baselines

**Status labels:** VERIFIED run. **This is a traffic-classification and scale demo. These labels are not cryptographic or PQC labels, and this model says nothing about quantum readiness.**

TEST/demo context: the demo bank in `demo_bank_app/` is SYNTHETIC.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
DERIVED.mkdir(parents=True, exist_ok=True)
# This notebook reads no credentials. If a future cell needs one, read it from an environment variable at runtime
# and never display it.

In [ ]:
from cryptoshift.datasets import resolve_dataset_paths
from cryptoshift.ml import cesnet_service
h5 = resolve_dataset_paths().cesnet_h5_path
res = cesnet_service.run(h5, chunks_per_day=15, chunk_rows=1000)
res.pop('_model')
print(res['disclaimer'])
{k: res[k] for k in ('sample','ppi_rows_identified','model','seed','library_versions')}

In [ ]:
import pandas as pd
pd.DataFrame({k: {m: v[m] for m in ('accuracy','macro_f1')} for k, v in res['metrics'].items()}).T

In [ ]:
pd.Series(res['metrics']['model']['per_class_recall']).sort_values().round(3)